# Text summarization with small files with Amazon Nova and Anthropic Claude

> *This notebook should work well with the **`Python 3`** kernel in SageMaker Studio*

> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

## Overview

In this example, you are going to ingest a small amount of data (String data) directly into Amazon Bedrock API (using Amazon Nova 2 Lite and Claude Sonnet 5.5) and give it an instruction to summarize the respective text.

### Architecture

![](../imgs/41-text-simple-1.png)

In this architecture:

1. A small piece of text (or small file) is loaded
1. A foundation model processes those data
1. Model returns a response with the summary of the ingested text

### Use case

This approach can be used to summarize call transcripts, meetings transcripts, books, articles, blog posts, and other relevant content.

### Challenges
This approach can be used when the input text or file fits within the model context length. In notebook `06_OpenSource_examples/00_Langchain_TextGeneration_examples/04_long text summarization using LCEL chains on Langchain.ipynb`, we will explore an approach to address the challenge when users have large document(s) that exceed the token limit.

## Setup

⚠️ ⚠️ ⚠️ Before running this notebook, ensure you've run the [Bedrock boto3 setup notebook](../00_Intro/bedrock_boto3_setup.ipynb#Prerequisites) notebook. ⚠️ ⚠️ ⚠️

In [1]:
import json
import os
import sys

import boto3
import botocore

boto3_bedrock = boto3.client('bedrock-runtime')

## Summarizing a short text with boto3
 
To learn detail of API request to Amazon Bedrock, this notebook introduces how to create API request and send the request via Boto3 rather than relying on langchain, which gives simpler API by wrapping Boto3 operation. 

### Request Syntax of Converse in Boto3

We use the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) for sending a request to a foundation model. The request has the same shape for every provider: a list of `messages` and an `inferenceConfig`. Inference parameters are:
- **maxTokens** configures the max number of tokens to use in the generated response
- **temperature** controls randomness: lower values produce more deterministic output
- **topP** controls token sampling diversity
- **stopSequences** stops the generation when any of these strings appears

Amazon Nova 2 Lite ([model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html)) accepts all of them. Claude 4.7+ only accepts `maxTokens` and `stopSequences` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)).

In [2]:
prompt = """
Please provide a summary of the following text. Do not add any information that is not mentioned in the text below.

<text>
AWS took all of that feedback from customers, and today we are excited to announce Amazon Bedrock, \
a new service that makes FMs from AI21 Labs, Anthropic, Stability AI, and Amazon accessible via an API. \
Bedrock is the easiest way for customers to build and scale generative AI-based applications using FMs, \
democratizing access for all builders. Bedrock will offer the ability to access a range of powerful FMs \
for text and images—including Amazons Titan FMs, which consist of two new LLMs we’re also announcing \
today—through a scalable, reliable, and secure AWS managed service. With Bedrock’s serverless experience, \
customers can easily find the right model for what they’re trying to get done, get started quickly, privately \
customize FMs with their own data, and easily integrate and deploy them into their applications using the AWS \
tools and capabilities they are familiar with, without having to manage any infrastructure (including integrations \
with Amazon SageMaker ML features like Experiments to test different models and Pipelines to manage their FMs at scale).
</text>

"""

## Creating request body with prompt and inference parameters 

Following the request syntax of `converse`, you create the messages and inference parameters with the above prompt.

### Amazon Nova 2 Lite:

In [3]:
messages = [{"role": "user", "content": [{"text": prompt}]}]
inference_config = {
    "maxTokens": 1024,
    "stopSequences": [],
    "temperature": 0,
    "topP": 1,
}

## Invoke foundation model via Boto3

Here sends the API request to Amazon Bedrock with specifying request parameters `modelId`, `accept`, and `contentType`. Following the prompt, the foundation model in Amazon Bedrock sumamrizes the text.

In [4]:
modelId = "us.amazon.nova-2-lite-v1:0"

try:

    response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=inference_config)

    print(response["output"]["message"]["content"][0]["text"])

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

### Summary

Amazon Bedrock, a new AWS service, is designed to make foundation models (FMs) from AI21 Labs, Anthropic, Stability AI, and Amazon accessible through an API. It simplifies the process for developers to build and scale generative AI applications using these FMs. Bedrock provides access to powerful models for text and images, including Amazon’s new Titan FMs—two newly announced large language models (LLMs). 

The service offers a serverless experience, allowing users to quickly find the appropriate model, privately customize it with their own data, and integrate it into their applications using familiar AWS tools. Bedrock eliminates the need to manage infrastructure and integrates with Amazon SageMaker features such as Experiments for testing models and Pipelines for managing FMs at scale. It aims to democratize access to advanced AI models for all developers.


In the above the TTP generates the entire summary for the given prompt in a single output. Note that this can be slow if the output contains large amount of tokens. 


___

### Claude Sonnet 5.5 (Messages API through InvokeModel):

Claude 4.7+ does not accept `temperature`, `top_k` or `top_p` Claude 5.5 has [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) on by default: the response can start with a `thinking` block (Invoke) or a `reasoningContent` block (Converse) before the `text` block, so the code below picks the text blocks explicitly instead of reading `content[0]`. These demos ask for short answers with a small token budget, so [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) is turned off explicitly with `"thinking": {"type": "disabled"}`; otherwise the model can spend the whole `max_tokens` budget thinking and return no text. Claude Sonnet 5.5 rejects `"disabled"` and uses `"thinking": {"type": "between_tools"}` instead, which without tools returns only text (see the per-model table in the [thinking docs](https://platform.claude.com/docs/en/build-with-claude/thinking)). ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)).

In [5]:
prompt = """
Please provide a summary of the following text.
<text>
AWS took all of that feedback from customers, and today we are excited to announce Amazon Bedrock, \
a new service that makes FMs from AI21 Labs, Anthropic, Stability AI, and Amazon accessible via an API. \
Bedrock is the easiest way for customers to build and scale generative AI-based applications using FMs, \
democratizing access for all builders. Bedrock will offer the ability to access a range of powerful FMs \
for text and images—including Amazons Titan FMs, which consist of two new LLMs we’re also announcing \
today—through a scalable, reliable, and secure AWS managed service. With Bedrock’s serverless experience, \
customers can easily find the right model for what they’re trying to get done, get started quickly, privately \
customize FMs with their own data, and easily integrate and deploy them into their applications using the AWS \
tools and capabilities they are familiar with, without having to manage any infrastructure (including integrations \
with Amazon SageMaker ML features like Experiments to test different models and Pipelines to manage their FMs at scale).
</text>

"""

In [6]:
body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 4096,
    "thinking": {"type": "between_tools"},
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ],
})

In [7]:
modelId = "global.anthropic.claude-sonnet-5-5"
accept = 'application/json'
contentType = 'application/json'

response = boto3_bedrock.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
response_body = json.loads(response.get('body').read())
print("".join(block["text"] for block in response_body["content"] if block["type"] == "text"))

AWS announced Amazon Bedrock, a new managed service that gives customers API access to foundation models (FMs) from AI21 Labs, Anthropic, Stability AI, and Amazon, including two new Amazon Titan large language models. It is designed to be the easiest way to build and scale generative AI applications, making these models available to all builders.

Its serverless experience lets customers find the right model for their task, get started quickly, and privately customize models with their own data. They can then integrate and deploy models using familiar AWS tools without managing infrastructure. Bedrock also integrates with Amazon SageMaker features such as Experiments, for testing different models, and Pipelines, for managing FMs at scale.


## Conclusion
You have now experimented with using `boto3` SDK which provides a vanilla exposure to Amazon Bedrock API. Using this API you have seen the use case of generating a summary of AWS news about Amazon Bedrock.

### Take aways
- Adapt this notebook to experiment with different models available through Amazon Bedrock such as Anthropic Claude and Meta Llama models.
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.